In [0]:
CREATE OR REPLACE TABLE dbx_joshdevph_dev.mart.fact_daily_yellow_taxi_trips
USING DELTA
AS

SELECT
    -- ============================================================
    -- Dimension Keys
    -- ============================================================
    CAST(DATE_FORMAT(tpep_pickup_datetime, 'yyyyMMdd') AS INT)
        AS date_key,

    PULocationID,
    DOLocationID,

    -- ============================================================
    -- Trip Measures
    -- ============================================================
    COUNT(*) AS total_trips,
    SUM(passenger_count) AS total_passenger_count,
    AVG(passenger_count) AS avg_passenger_count,
    SUM(trip_distance) AS total_trip_distance,
    AVG(trip_distance) AS avg_trip_distance,
    AVG(
        TIMESTAMPDIFF(
            MINUTE,
            tpep_pickup_datetime,
            tpep_dropoff_datetime
        )
    ) AS avg_trip_duration_minutes,

    -- ============================================================
    -- Fare Measures
    -- ============================================================
    SUM(fare_amount) AS total_fare_amount,
    AVG(fare_amount) AS avg_fare_amount,

    -- ============================================================
    -- Tip Measures
    -- ============================================================
    SUM(tip_amount) AS total_tip_amount,
    AVG(tip_amount) AS avg_tip_amount,

    -- ============================================================
    -- Toll Measures
    -- ============================================================
    SUM(tolls_amount) AS total_tolls_amount,
    AVG(tolls_amount) AS avg_tolls_amount,

    -- ============================================================
    -- Tax Measures
    -- ============================================================
    SUM(mta_tax) AS total_mta_tax,
    AVG(mta_tax) AS avg_mta_tax,

    -- ============================================================
    -- Other Fees / Surcharges
    -- ============================================================
    SUM(extra) AS total_extra,
    AVG(extra) AS avg_extra,
    SUM(improvement_surcharge) AS total_improvement_surcharge,
    AVG(improvement_surcharge) AS avg_improvement_surcharge,
    SUM(congestion_surcharge) AS total_congestion_surcharge,
    AVG(congestion_surcharge) AS avg_congestion_surcharge,
    SUM(Airport_fee) AS total_airport_fee,
    AVG(Airport_fee) AS avg_airport_fee,
    SUM(cbd_congestion_fee) AS total_cbd_congestion_fee,
    AVG(cbd_congestion_fee) AS avg_cbd_congestion_fee,

    -- ============================================================
    -- Overall Amount Measures
    -- ============================================================
    SUM(total_amount) AS total_amount,
    AVG(total_amount) AS avg_total_amount,

    -- ============================================================
    -- Derived Measures
    -- ============================================================
    AVG(
        CASE
            WHEN fare_amount > 0
            THEN (tip_amount / fare_amount) * 100
        END
    ) AS avg_tip_percentage,
    AVG(
        CASE
            WHEN trip_distance > 0
            THEN fare_amount / trip_distance
        END
    ) AS avg_fare_per_mile,
    AVG(
        CASE
            WHEN passenger_count > 0
            THEN total_amount / passenger_count
        END
    ) AS avg_amount_per_passenger

FROM dbx_joshdevph_dev.processed.vw_yellow_tripdata_valid

GROUP BY
    CAST(DATE_FORMAT(tpep_pickup_datetime, 'yyyyMMdd') AS INT),
    PULocationID,
    DOLocationID;